# 03 — Aggregations: column lineage с GROUP BY

GROUP BY порождает column lineage с двумя видами edges (per OL spec — `type` × `subtype`):

- **DIRECT edges** на конкретные выходные колонки:
  - `customer_id` (group key) → `raw_orders.customer_id` с subtype `IDENTITY`
  - `total_spent`, `avg_check` → `raw_orders.amount` с subtype `AGGREGATION`
  - `first_order_ts`, `last_order_ts` → `raw_orders.order_ts` с subtype `AGGREGATION`
- **INDIRECT/GROUP_BY edges** — отдельная штука: per [OL Spark column lineage docs](https://openlineage.io/docs/integrations/spark/spark_column_lineage), агрегация порождает ребро от group-key колонки (`raw_orders.customer_id`) ко **всем** выходным колонкам с subtype `GROUP_BY`. Это значит, что в Marquez у `total_spent` будет видно две входящие стрелки: DIRECT/AGGREGATION от `amount` и INDIRECT/GROUP_BY от `customer_id`.

**Что построим:** `default.agg_customer_stats` с per-customer метриками.

Prerequisite: прогнан `00_setup.ipynb`. **Restart Jupyter kernel** перед запуском.

In [1]:
try:
    spark.stop()
except NameError:
    pass

from pyspark.sql import SparkSession
spark = (
    SparkSession.builder
    .appName('lineage_03_aggregations')
    .enableHiveSupport()
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
assert spark.sparkContext.appName == 'lineage_03_aggregations', \
    f'appName fallback to {spark.sparkContext.appName!r} — restart Jupyter kernel'
print('app name:', spark.sparkContext.appName)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/27 13:08:55 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/27 13:08:56 WARN Client: Neither spark.yarn.jars nor spark.yarn.archive is set, falling back to uploading libraries under SPARK_HOME.
26/05/27 13:09:04 WARN BlackbirdModule: Unable to find Java 9+ MethodHandles.privateLookupIn.  Blackbird is not performing optimally!
26/05/27 13:09:05 WARN SparkApplicationNameApplicationJobNameProvider: Failed to obtain the application name. Using the default value [unknown]. Set the [spark.app.name] or [spark.openlineage.appName] property.
26/05/27 13:09:05 WARN SparkApplicationNameApplicationJobNameProvider: Failed to obtain the application name. Using the default value [unknown]. Set the [spark.app.name] or [spark.openlineage.appName] property.


app name: lineage_03_aggregations


In [2]:
df = spark.sql('''
    SELECT
        customer_id,
        count(*)        AS orders_count,
        sum(amount)     AS total_spent,
        avg(amount)     AS avg_check,
        min(order_ts)   AS first_order_ts,
        max(order_ts)   AS last_order_ts
    FROM default.raw_orders
    GROUP BY customer_id
''')
df.write.mode('overwrite').saveAsTable('default.agg_customer_stats')
spark.table('default.agg_customer_stats').orderBy('customer_id').show(5, truncate=False)

26/05/27 13:09:08 WARN HiveConf: HiveConf of name hive.metastore.event.db.notification.api.auth does not exist
26/05/27 13:09:08 WARN HiveConf: HiveConf of name hive.log.dir does not exist
26/05/27 13:09:08 WARN HiveConf: HiveConf of name hive.root.logger does not exist
26/05/27 13:09:08 WARN HiveClientImpl: Detected HiveConf hive.execution.engine is 'tez' and will be reset to 'mr' to disable useless hive logic
26/05/27 13:09:12 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.


+-----------+------------+------------------+------------------+-------------------+-------------------+
|customer_id|orders_count|total_spent       |avg_check         |first_order_ts     |last_order_ts      |
+-----------+------------+------------------+------------------+-------------------+-------------------+
|0          |5           |2810.926834750286 |562.1853669500572 |2023-11-14 22:13:20|2023-11-15 04:53:20|
|1          |5           |1873.2587614025167|374.65175228050333|2023-11-14 22:14:20|2023-11-15 04:54:20|
|2          |5           |2645.14552754322  |529.029105508644  |2023-11-14 22:15:20|2023-11-15 04:55:20|
|3          |5           |1151.0686958279296|230.2137391655859 |2023-11-14 22:16:20|2023-11-15 04:56:20|
|4          |5           |2655.2608749794836|531.0521749958967 |2023-11-14 22:17:20|2023-11-15 04:57:20|
+-----------+------------+------------------+------------------+-------------------+-------------------+
only showing top 5 rows



In [3]:
spark.stop()

## Что смотреть в Marquez

**DIRECT edges (явный источник колонки):**
- `customer_id` ← `raw_orders.customer_id` — subtype `IDENTITY` (group key переносится 1:1).
- `total_spent`, `avg_check` ← `raw_orders.amount` — subtype `AGGREGATION`.
- `first_order_ts`, `last_order_ts` ← `raw_orders.order_ts` — subtype `AGGREGATION`.
- `orders_count` — `count(*)` обычно не имеет конкретной входной колонки, у этой output column DIRECT-стрелок может не быть вообще.

**INDIRECT/GROUP_BY edges (контекст агрегации):**
- На **каждую** выходную колонку (включая `total_spent`, `orders_count` и сам `customer_id`) добавится INDIRECT-стрелка от `raw_orders.customer_id` с subtype `GROUP_BY` — это фиксирует, что результирующее значение зависит от группировки.